# STG_5.3 — Dataset de entrenamiento con features de autoría (solo 2019+)

Construye `df_entrenamiento_autor.csv` a partir de:
- `data/df_modelado_autor.csv` — 28.738 votos consolidados + features de autoría (specs 011/012)
- `data/df_features_titulo.csv` — embeddings semánticos de los 1.022 títulos únicos

Es la versión "con autoría y recortada a 2019+" de `df_entrenamiento.csv` (spec 009).

**Reglas de oro (spec 013):**
- El historial de cada diputado se calcula con **todos los años disponibles** (incluye pre-2019).
- El filtro a **2019 en adelante** se aplica **al final**, después de calcular las features.
- Las 4 votaciones anómalas (id 1925, 3527, 3585, 3494) no se excluyen del historial —
  solo no deben quedar como filas de entrenamiento (ya lo garantiza el filtro de año, se verifica igual).

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Rutas
DATA_DIR = Path("../data")


## T1 — Carga de datos

In [2]:
# T1 — Carga de datos
df_votos = pd.read_csv(DATA_DIR / "df_modelado_autor.csv", parse_dates=["fecha_votacion"])
df_titulos = pd.read_csv(DATA_DIR / "df_features_titulo.csv")

print(f"df_modelado_autor → {len(df_votos):,} filas | {df_votos['diputado'].nunique()} diputados únicos")
print(f"df_titulos        → {len(df_titulos):,} filas | {df_titulos['titulo_base'].nunique()} títulos únicos")
print()
print("Columnas df_modelado_autor:", list(df_votos.columns))
print()
print("Rango de fechas:", df_votos['fecha_votacion'].min().date(), "→", df_votos['fecha_votacion'].max().date())
print()
print("Distribución de votos:")
print(df_votos['voto'].value_counts())


df_modelado_autor → 28,738 filas | 259 diputados únicos
df_titulos        → 1,022 filas | 1022 títulos únicos

Columnas df_modelado_autor: ['diputado', 'titulo_base', 'voto', 'bloque', 'provincia', 'fecha_votacion', 'id_votacion', 'fuente_consolidacion', 'autor_final', 'camara_origen', 'fuente_autor', 'score_fuzzy', 'bloque_autor', 'fuente_bloque_autor', 'es_poder_ejecutivo', 'es_oficialista', 'coincide_bloque_autor', 'es_oficialista_b', 'coincide_bloque_autor_b']

Rango de fechas: 1993-12-22 → 2026-05-20

Distribución de votos:
voto
AFIRMATIVO    19356
NEGATIVO       4840
AUSENTE        3656
ABSTENCION      716
ABSTENCIÓN      170
Name: count, dtype: int64


In [3]:
# Verificaciones T1
assert len(df_votos) == 28738, f"Se esperaban 28.738 filas, hay {len(df_votos)}"
assert df_titulos['titulo_base'].nunique() == 1022, "Se esperaban 1.022 títulos únicos"
assert df_votos['fecha_votacion'].isna().sum() == 0, "Hay NaN en fecha_votacion"
assert 'id_votacion' in df_votos.columns, "Falta id_votacion (necesario para excluir los 4 ids anómalos)"
for col in ['bloque_autor', 'es_poder_ejecutivo', 'es_oficialista', 'coincide_bloque_autor',
            'es_oficialista_b', 'coincide_bloque_autor_b']:
    assert col in df_votos.columns, f"Falta la feature de autoría {col}"

print("✓ T1 PASA: 28.738 filas, 1.022 títulos, id_votacion y features de autoría presentes.")


✓ T1 PASA: 28.738 filas, 1.022 títulos, id_votacion y features de autoría presentes.


## T2 — Filtrar AUSENTE y unificar ABSTENCIÓN

In [4]:
# T2 — Los votos AUSENTE no representan posición política → se filtran
# ABSTENCION y ABSTENCIÓN son el mismo voto → se unifican (idéntico a STG_5)
antes = len(df_votos)
df_votos = df_votos[df_votos['voto'] != 'AUSENTE'].copy()
df_votos['voto'] = df_votos['voto'].replace('ABSTENCION', 'ABSTENCIÓN')

print(f"Filas eliminadas (AUSENTE): {antes - len(df_votos):,}")
print(f"Filas restantes           : {len(df_votos):,}")
print()
print("Distribución final de votos:")
print(df_votos['voto'].value_counts().to_string())


Filas eliminadas (AUSENTE): 3,656
Filas restantes           : 25,082

Distribución final de votos:
voto
AFIRMATIVO    19356
NEGATIVO       4840
ABSTENCIÓN      886


In [5]:
# Verificaciones T2
assert (df_votos['voto'] == 'AUSENTE').sum() == 0, "Quedaron votos AUSENTE"
assert set(df_votos['voto'].unique()) == {'AFIRMATIVO', 'NEGATIVO', 'ABSTENCIÓN'},     f"Valores de voto inesperados: {df_votos['voto'].unique()}"
assert len(df_votos) == 28738 - 3656, f"Se esperaban {28738-3656} filas, hay {len(df_votos)}"

print(f"✓ T2 PASA: {len(df_votos):,} filas, solo AFIRMATIVO/NEGATIVO/ABSTENCIÓN.")


✓ T2 PASA: 25,082 filas, solo AFIRMATIVO/NEGATIVO/ABSTENCIÓN.


## T3 — Merge de embeddings y tema_id

In [6]:
# T3 — Unir tema_id, tema_label y embeddings desde df_features_titulo
# NOTA: df_modelado_autor.csv NO contiene tema_id; viene solo de df_features_titulo.csv
emb_cols = [c for c in df_titulos.columns if c.startswith('emb_')]
print(f"Columnas de embeddings: {len(emb_cols)}")

df = df_votos.merge(
    df_titulos[['titulo_base', 'tema_id', 'tema_label'] + emb_cols],
    on='titulo_base',
    how='left'
)

print(f"Filas tras merge: {len(df):,}")
print(f"NaN en embeddings: {df[emb_cols].isna().sum().sum()}")
print(f"NaN en tema_id   : {df['tema_id'].isna().sum()}")


Columnas de embeddings: 384


Filas tras merge: 25,082

NaN en embeddings: 0
NaN en tema_id   : 0


In [7]:
# Verificaciones T3
assert len(df) == len(df_votos), "El merge cambió la cantidad de filas"
assert df[emb_cols].isna().sum().sum() == 0, "Hay NaN en los embeddings tras el merge"
assert df['tema_id'].isna().sum() == 0, "Hay NaN en tema_id tras el merge"

print("✓ T3 PASA: merge correcto, 0 NaN en embeddings y tema_id.")


✓ T3 PASA: merge correcto, 0 NaN en embeddings y tema_id.


## T4 — Features históricas del diputado (historial COMPLETO, sin leakage)

Reusa la misma lógica de STG_5 (`media_acumulada_pasado`). A diferencia de STG_5, acá el
historial se calcula sobre **todos los años disponibles** (incluye pre-2019); el filtro a
2019+ se aplica recién en T9, después de tener todas las features listas. Así cada voto de
2019 en adelante "recuerda" lo que el diputado votó antes, sin perder memoria.

In [8]:
# es_afirmativo: 1 si el diputado votó AFIRMATIVO, 0 si no
df['es_afirmativo'] = (df['voto'] == 'AFIRMATIVO').astype(float)

# voto_bloque: para cada (titulo, fecha, bloque), el voto más frecuente del bloque
voto_bloque = (
    df.groupby(['titulo_base', 'fecha_votacion', 'bloque'])['voto']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
    .rename(columns={'voto': 'voto_bloque'})
)
df = df.merge(voto_bloque, on=['titulo_base', 'fecha_votacion', 'bloque'], how='left')

# alineado_bloque: 1 si el diputado votó igual que la moda de su bloque
df['alineado_bloque'] = (df['voto'] == df['voto_bloque']).astype(float)

print(f"es_afirmativo   — media: {df['es_afirmativo'].mean():.3f} | NaN: {df['es_afirmativo'].isna().sum()}")
print(f"alineado_bloque — media: {df['alineado_bloque'].mean():.3f} | NaN: {df['alineado_bloque'].isna().sum()}")

assert df['es_afirmativo'].isna().sum() == 0
assert df['alineado_bloque'].isna().sum() == 0
assert len(df) == 25082


es_afirmativo   — media: 0.772 | NaN: 0
alineado_bloque — media: 0.982 | NaN: 0


In [9]:
def media_acumulada_pasado(serie):
    """
    Para cada posicion i, devuelve la media de los valores 0..i-1.
    Posicion 0 devuelve NaN (no hay historia previa).
    Nunca incluye el valor actual -> cero leakage.
    """
    cumsum   = serie.cumsum().shift(1)
    cumcount = serie.notna().cumsum().shift(1)
    return cumsum / cumcount

# Ordenar por diputado y fecha (obligatorio para que shift opere en orden temporal)
df = df.sort_values(['diputado', 'fecha_votacion']).reset_index(drop=True)

# tasa_afirmativo_hist: % de votos AFIRMATIVO del diputado en toda su historia previa
df['tasa_afirmativo_hist'] = (
    df.groupby('diputado')['es_afirmativo']
    .transform(media_acumulada_pasado)
)

# tasa_afirmativo_tema_hist: idem pero solo para el tema_id especifico
df['tasa_afirmativo_tema_hist'] = (
    df.groupby(['diputado', 'tema_id'])['es_afirmativo']
    .transform(media_acumulada_pasado)
)

# tasa_alineacion_bloque_hist: % de veces que el diputado voto igual que su bloque
df['tasa_alineacion_bloque_hist'] = (
    df.groupby('diputado')['alineado_bloque']
    .transform(media_acumulada_pasado)
)

for col in ['tasa_afirmativo_hist', 'tasa_afirmativo_tema_hist', 'tasa_alineacion_bloque_hist']:
    nan_count = df[col].isna().sum()
    media = df[col].mean()
    print(f"{col}: media={media:.3f} | NaN={nan_count:,}")


tasa_afirmativo_hist: media=0.773 | NaN=259
tasa_afirmativo_tema_hist: media=0.767 | NaN=2,541
tasa_alineacion_bloque_hist: media=0.984 | NaN=259


In [10]:
# Fechas de corte (idénticas a STG_5)
CORTE_2023 = pd.Timestamp('2023-12-10')  # inicio del gobierno actual
CORTE_2026 = pd.Timestamp('2026-01-01')  # inicio de 2026

for col_out, corte in [
    ('tasa_afirmativo_desde_2023', CORTE_2023),
    ('tasa_afirmativo_2026',       CORTE_2026),
]:
    col_mask  = f'_mask_{col_out}'
    col_afirm = f'_afirm_{col_out}'

    df[col_mask]  = (df['fecha_votacion'] >= corte).astype(float)
    df[col_afirm] = df['es_afirmativo'] * df[col_mask]

    cumsum_afirm = df.groupby('diputado')[col_afirm].transform(lambda x: x.cumsum().shift(1))
    cumsum_mask  = df.groupby('diputado')[col_mask].transform(lambda x: x.cumsum().shift(1))

    df[col_out] = cumsum_afirm / cumsum_mask

    df.drop(columns=[col_mask, col_afirm], inplace=True)

for col in ['tasa_afirmativo_desde_2023', 'tasa_afirmativo_2026']:
    non_nan = df[col].notna().sum()
    print(f"{col}: media={df[col].mean():.3f} | NaN={df[col].isna().sum():,} | no-NaN={non_nan:,}")

print()
for col, corte in [('tasa_afirmativo_desde_2023', CORTE_2023), ('tasa_afirmativo_2026', CORTE_2026)]:
    primeros_en_ventana = (
        df[df['fecha_votacion'] >= corte]
        .groupby('diputado')
        .head(1)
    )
    assert primeros_en_ventana[col].isna().all(), f"LEAKAGE en {col}"

print("Primer voto de cada diputado en cada ventana es NaN (correcto).")


tasa_afirmativo_desde_2023: media=0.646 | NaN=12,730 | no-NaN=12,352
tasa_afirmativo_2026: media=0.764 | NaN=21,278 | no-NaN=3,804

Primer voto de cada diputado en cada ventana es NaN (correcto).


In [11]:
# cumcount() empieza en 0 para el primer voto de cada diputado
df['n_votos_hist'] = df.groupby('diputado').cumcount()

print(f"n_votos_hist — min={df['n_votos_hist'].min()} | max={df['n_votos_hist'].max()} | NaN={df['n_votos_hist'].isna().sum()}")
assert df['n_votos_hist'].isna().sum() == 0
assert df['n_votos_hist'].min() == 0


n_votos_hist — min=0 | max=435 | NaN=0


In [12]:
# Riesgo resuelto: si hicieramos shift(1) fila por fila dentro del grupo (bloque, tema_id),
# el 'anterior' podria ser otro diputado votando EL MISMO DIA (no el dia anterior).
# Solucion: agregar primero por dia, hacer el shift a nivel de dia, y pegar de vuelta.

daily = (
    df.groupby(['bloque', 'tema_id', 'fecha_votacion'])['es_afirmativo']
    .agg(suma='sum', n='count')
    .reset_index()
    .sort_values(['bloque', 'tema_id', 'fecha_votacion'])
)

daily['suma_acum'] = daily.groupby(['bloque', 'tema_id'])['suma'].transform(
    lambda x: x.cumsum().shift(1)
)
daily['n_acum'] = daily.groupby(['bloque', 'tema_id'])['n'].transform(
    lambda x: x.cumsum().shift(1)
)
daily['tasa_afirmativo_bloque_tema'] = daily['suma_acum'] / daily['n_acum']

df = df.merge(
    daily[['bloque', 'tema_id', 'fecha_votacion', 'tasa_afirmativo_bloque_tema']],
    on=['bloque', 'tema_id', 'fecha_votacion'],
    how='left'
)

nan_count = df['tasa_afirmativo_bloque_tema'].isna().sum()
print(f"tasa_afirmativo_bloque_tema: media={df['tasa_afirmativo_bloque_tema'].mean():.3f} | NaN={nan_count:,}")
assert nan_count > 0, "Debería haber NaN para los primeros votos de cada bloque-tema"
assert len(df) == 25082, f"El merge cambió la cantidad de filas: {len(df)}"

print()
print("✓ T4 PASA: 9 features históricas calculadas sobre el historial completo (todos los años).")


tasa_afirmativo_bloque_tema: media=0.725 | NaN=2,906

✓ T4 PASA: 9 features históricas calculadas sobre el historial completo (todos los años).


## T5 — Chequeo automático de no-leakage (explícito, sobre TODOS los años)

Se verifica **antes** de recortar a 2019+: el primer voto de cada diputado (y de cada
diputado-tema) debe tener las features históricas en NaN. Si alguna no lo es, hay fuga de
información y la notebook frena acá mismo.

In [13]:
# Features que deben ser NaN en el primer voto de cada diputado.
FEATURES_HIST = [
    'tasa_afirmativo_hist',
    'tasa_alineacion_bloque_hist',
    'tasa_afirmativo_desde_2023',
    'tasa_afirmativo_2026',
]

primeros_votos = df.groupby('diputado').head(1)
errores = []

for col in FEATURES_HIST:
    n_no_nan = primeros_votos[col].notna().sum()
    if n_no_nan > 0:
        errores.append(f"  LEAKAGE en '{col}': {n_no_nan} diputados tienen valor en su primer voto")

# tasa_afirmativo_tema_hist: el primer voto de cada (diputado, tema_id) debe ser NaN
primeros_tema = df.groupby(['diputado', 'tema_id']).head(1)
n_no_nan_tema = primeros_tema['tasa_afirmativo_tema_hist'].notna().sum()
if n_no_nan_tema > 0:
    errores.append(f"  LEAKAGE en 'tasa_afirmativo_tema_hist': {n_no_nan_tema} combinaciones diputado-tema tienen valor en su primer voto")

if errores:
    raise AssertionError("CHEQUEO DE NO-LEAKAGE FALLIDO:\n" + "\n".join(errores))

print("=" * 60)
print("CHEQUEO DE NO-LEAKAGE (todos los años, antes de filtrar a 2019+): TODO PASA")
print("=" * 60)
for col in FEATURES_HIST:
    n_nan = primeros_votos[col].isna().sum()
    print(f"  {col}: {n_nan}/{len(primeros_votos)} primeros votos son NaN (correcto)")
print(f"  tasa_afirmativo_tema_hist: {primeros_tema['tasa_afirmativo_tema_hist'].isna().sum()}/{len(primeros_tema)} primeros votos por tema son NaN (correcto)")
print("=" * 60)
print()
print("✓ T5 PASA.")


CHEQUEO DE NO-LEAKAGE (todos los años, antes de filtrar a 2019+): TODO PASA
  tasa_afirmativo_hist: 259/259 primeros votos son NaN (correcto)
  tasa_alineacion_bloque_hist: 259/259 primeros votos son NaN (correcto)
  tasa_afirmativo_desde_2023: 259/259 primeros votos son NaN (correcto)
  tasa_afirmativo_2026: 259/259 primeros votos son NaN (correcto)
  tasa_afirmativo_tema_hist: 2541/2541 primeros votos por tema son NaN (correcto)

✓ T5 PASA.


## T6 — Cascada de relleno de NaN (honesta, sin leakage)

Cada feature cae a la más general disponible. En último recurso: `0.5` neutro (NO la media
global del dataset, que miraría el futuro). `n_votos_hist` ya tiene 0 NaN; no necesita relleno.

In [14]:
ANTES = {col: df[col].isna().sum() for col in [
    'tasa_afirmativo_hist', 'tasa_afirmativo_tema_hist',
    'tasa_alineacion_bloque_hist', 'tasa_afirmativo_desde_2023',
    'tasa_afirmativo_2026', 'tasa_afirmativo_bloque_tema'
]}

# 1. tema → hist individual (cuando no hay historia en ese tema, usar la tasa global del diputado)
df['tasa_afirmativo_tema_hist'] = df['tasa_afirmativo_tema_hist'].fillna(df['tasa_afirmativo_hist'])

# 2. ventanas → hist individual (cuando no hay votos en la ventana, usar la tasa global)
df['tasa_afirmativo_desde_2023'] = df['tasa_afirmativo_desde_2023'].fillna(df['tasa_afirmativo_hist'])
df['tasa_afirmativo_2026']       = df['tasa_afirmativo_2026'].fillna(df['tasa_afirmativo_hist'])

# 3. bloque-tema → hist individual (cuando el bloque no tiene historia en ese tema)
df['tasa_afirmativo_bloque_tema'] = df['tasa_afirmativo_bloque_tema'].fillna(df['tasa_afirmativo_hist'])

# 4. Residual → 0.5 neutro (primerísimos votos donde no hay ninguna historia previa)
FEATURES_HIST_MODELADO = [
    'tasa_afirmativo_hist', 'tasa_afirmativo_tema_hist',
    'tasa_alineacion_bloque_hist', 'tasa_afirmativo_desde_2023',
    'tasa_afirmativo_2026', 'tasa_afirmativo_bloque_tema'
]
for col in FEATURES_HIST_MODELADO:
    df[col] = df[col].fillna(0.5)

print("NaN antes y después del relleno:")
print(f"{'Feature':<35} {'Antes':>7} {'Después':>8}")
print("-" * 52)
for col in FEATURES_HIST_MODELADO:
    despues = df[col].isna().sum()
    print(f"{col:<35} {ANTES[col]:>7,} {despues:>8,}")

assert all(df[col].isna().sum() == 0 for col in FEATURES_HIST_MODELADO), "NaN residuales tras el relleno"
print()
print("✓ T6 PASA: 0 NaN residuales en todas las features históricas.")


NaN antes y después del relleno:
Feature                               Antes  Después
----------------------------------------------------
tasa_afirmativo_hist                    259        0
tasa_afirmativo_tema_hist             2,541        0
tasa_alineacion_bloque_hist             259        0
tasa_afirmativo_desde_2023           12,730        0
tasa_afirmativo_2026                 21,278        0
tasa_afirmativo_bloque_tema           2,906        0

✓ T6 PASA: 0 NaN residuales en todas las features históricas.


## T7 — Normalizar y codificar `bloque_autor` + features binarias de autor

`bloque_autor` trae la misma coalición escrita de más de una forma (ej. "Frente De Todos" vs.
"FRENTE DE TODOS", "La Libertad Avanza" vs. "LA LIBERTAD AVANZA", con y sin tilde). Se
normaliza con la misma convención que ya usa el proyecto para nombres de diputados: sin
tildes, en mayúsculas, espacios colapsados. Esto **no** intenta unificar bloques que son
genuinamente distintos (ej. "Ucr" vs. "Ucr - Union Civica Radical" siguen siendo valores
distintos): solo colapsa diferencias de grafía de la misma etiqueta.

In [15]:
import unicodedata
import re

def normalizar_texto(s):
    """Sin tildes, mayúsculas, espacios colapsados (misma convención que match de diputados)."""
    if pd.isna(s):
        return s
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii')
    s = s.upper().strip()
    s = re.sub(r'\s+', ' ', s)
    return s

antes_unicos = df['bloque_autor'].nunique()
df['bloque_autor_norm'] = df['bloque_autor'].apply(normalizar_texto)
despues_unicos = df['bloque_autor_norm'].nunique()

print(f"bloque_autor: {antes_unicos} valores únicos antes de normalizar -> {despues_unicos} después")
print()
print("Ejemplos que colapsaron (mismo valor normalizado, distinta grafía original):")
colapsos = (
    df[['bloque_autor', 'bloque_autor_norm']]
    .drop_duplicates()
    .groupby('bloque_autor_norm')
    .filter(lambda g: len(g) > 1)
    .sort_values('bloque_autor_norm')
)
print(colapsos.to_string(index=False))


bloque_autor: 55 valores únicos antes de normalizar -> 51 después

Ejemplos que colapsaron (mismo valor normalizado, distinta grafía original):
         bloque_autor    bloque_autor_norm
     Coalición Civica     COALICION CIVICA
     Coalicion Civica     COALICION CIVICA
      Frente De Todos      FRENTE DE TODOS
      FRENTE DE TODOS      FRENTE DE TODOS
   LA LIBERTAD AVANZA   LA LIBERTAD AVANZA
   La Libertad Avanza   LA LIBERTAD AVANZA
 Unión Cívica Radical UNION CIVICA RADICAL
Unión Cívica Radical  UNION CIVICA RADICAL


In [16]:
from sklearn.preprocessing import OrdinalEncoder

enc_autor = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[['bloque_autor_enc']] = enc_autor.fit_transform(df[['bloque_autor_norm']])
joblib.dump(enc_autor, DATA_DIR / 'encoder_bloque_autor.joblib')

print(f"bloque_autor_norm codificado: {df['bloque_autor_norm'].nunique()} categorías -> códigos "
      f"{int(df['bloque_autor_enc'].min())} a {int(df['bloque_autor_enc'].max())}")
print("Encoder guardado en data/encoder_bloque_autor.joblib")

# Features binarias de autor: ya vienen calculadas de las specs 011/012, solo se verifican
FEATURES_AUTOR_BIN = ['es_poder_ejecutivo', 'es_oficialista', 'coincide_bloque_autor',
                       'es_oficialista_b', 'coincide_bloque_autor_b']
print()
for col in FEATURES_AUTOR_BIN:
    vals = sorted(df[col].unique())
    n_nan = df[col].isna().sum()
    print(f"{col:<26} valores={vals} | NaN={n_nan}")
    assert n_nan == 0, f"{col} tiene NaN (no debería, -1 es 'sin dato')"
    assert set(vals).issubset({-1, 0, 1}), f"{col} tiene valores fuera de {{-1,0,1}}: {vals}"

assert df['bloque_autor_enc'].isna().sum() == 0
print()
print("✓ T7 PASA: bloque_autor normalizado y codificado; 5 features binarias de autor verificadas.")


bloque_autor_norm codificado: 51 categorías -> códigos 0 a 50
Encoder guardado en data/encoder_bloque_autor.joblib

es_poder_ejecutivo         valores=[np.int64(-1), np.int64(0), np.int64(1)] | NaN=0
es_oficialista             valores=[np.int64(-1), np.int64(0), np.int64(1)] | NaN=0
coincide_bloque_autor      valores=[np.int64(-1), np.int64(0), np.int64(1)] | NaN=0
es_oficialista_b           valores=[np.int64(-1), np.int64(0), np.int64(1)] | NaN=0
coincide_bloque_autor_b    valores=[np.int64(-1), np.int64(0), np.int64(1)] | NaN=0

✓ T7 PASA: bloque_autor normalizado y codificado; 5 features binarias de autor verificadas.


## T8 — Encoding de `bloque` y `provincia` del votante

Igual que STG_5 (T10), pero el encoder se guarda con **nombre propio** para no pisar
`data/encoder_bloque_provincia.joblib` de la spec 009.

In [17]:
enc_votante = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[['bloque_enc', 'provincia_enc']] = enc_votante.fit_transform(df[['bloque', 'provincia']])

joblib.dump(enc_votante, DATA_DIR / 'encoder_bloque_provincia_autor.joblib')

print(f"Bloques distintos   : {df['bloque'].nunique()} -> códigos {int(df['bloque_enc'].min())} a {int(df['bloque_enc'].max())}")
print(f"Provincias distintas: {df['provincia'].nunique()} -> códigos {int(df['provincia_enc'].min())} a {int(df['provincia_enc'].max())}")
print(f"NaN en bloque_enc   : {df['bloque_enc'].isna().sum()}")
print(f"NaN en provincia_enc: {df['provincia_enc'].isna().sum()}")
print()
print("Encoder guardado en data/encoder_bloque_provincia_autor.joblib")

assert df['bloque_enc'].isna().sum() == 0
assert df['provincia_enc'].isna().sum() == 0
print()
print("✓ T8 PASA.")


Bloques distintos   : 55 -> códigos 0 a 54
Provincias distintas: 24 -> códigos 0 a 23
NaN en bloque_enc   : 0
NaN en provincia_enc: 0

Encoder guardado en data/encoder_bloque_provincia_autor.joblib

✓ T8 PASA.


## T9 — Filtrar a votaciones de 2019 en adelante

**Recién ahora** se recorta el dataset. Todas las features históricas (T4-T8) ya están
calculadas usando el historial completo (incluido 1993-2018); a partir de acá se descartan
esas filas como *casos a predecir*, pero ya cumplieron su función de alimentar la memoria de
cada diputado.

In [18]:
antes = len(df)
fecha_min_antes = df['fecha_votacion'].min()

df = df[df['fecha_votacion'] >= pd.Timestamp('2019-01-01')].copy()

print(f"Filas antes del filtro 2019+: {antes:,} (desde {fecha_min_antes.date()})")
print(f"Filas después del filtro    : {len(df):,} (desde {df['fecha_votacion'].min().date()})")
print(f"Filas eliminadas (pre-2019) : {antes - len(df):,}")
print()
print("Distribución de votos en 2019+:")
print(df['voto'].value_counts().to_string())

assert df['fecha_votacion'].min() >= pd.Timestamp('2019-01-01'), "Quedaron votos anteriores a 2019"
print()
print("✓ T9 PASA: dataset recortado a 2019 en adelante.")


Filas antes del filtro 2019+: 25,082 (desde 1993-12-22)
Filas después del filtro    : 20,608 (desde 2019-04-04)
Filas eliminadas (pre-2019) : 4,474

Distribución de votos en 2019+:
voto
AFIRMATIVO    15781
NEGATIVO       4148
ABSTENCIÓN      679

✓ T9 PASA: dataset recortado a 2019 en adelante.


## T10 — Verificar exclusión de ids anómalos y guardar `df_entrenamiento_autor.csv`

Verificación explícita (redundante con el filtro de año, pero deja probado el criterio de
aceptación de la spec): los 4 `id_votacion` anómalos no deben aparecer como filas de
entrenamiento. Después se arma y guarda el dataset final.

In [19]:
IDS_ANOMALOS = [1925, 3527, 3585, 3494]
presentes = df[df['id_votacion'].isin(IDS_ANOMALOS)]

print(f"Filas con id_votacion anómalo en el dataset de entrenamiento: {len(presentes)}")
assert len(presentes) == 0, f"Los ids anómalos {IDS_ANOMALOS} no deberían estar en el entrenamiento"

print("✓ Verificado: ninguno de los 4 ids anómalos (1925, 3527, 3585, 3494) quedó en el dataset.")


Filas con id_votacion anómalo en el dataset de entrenamiento: 0
✓ Verificado: ninguno de los 4 ids anómalos (1925, 3527, 3585, 3494) quedó en el dataset.


In [20]:
emb_cols = [c for c in df.columns if c.startswith('emb_')]

FEATURES_BASE = (
    ['tema_id'] +
    emb_cols +
    ['bloque_enc', 'provincia_enc'] +
    ['tasa_afirmativo_hist', 'tasa_afirmativo_tema_hist', 'tasa_alineacion_bloque_hist',
     'tasa_afirmativo_desde_2023', 'tasa_afirmativo_2026', 'n_votos_hist',
     'tasa_afirmativo_bloque_tema']
)

FEATURES_AUTOR = [
    'bloque_autor_enc', 'es_poder_ejecutivo',
    'es_oficialista', 'coincide_bloque_autor',           # escenario A (PRO fuera de LLA)
    'es_oficialista_b', 'coincide_bloque_autor_b',        # escenario B (PRO dentro de LLA)
]

TARGET = 'voto'
META = ['diputado', 'titulo_base', 'fecha_votacion', 'id_votacion', 'bloque', 'provincia', 'tema_label']

FEATURES = FEATURES_BASE + FEATURES_AUTOR
cols_guardar = META + FEATURES + [TARGET]
df_out = df[cols_guardar].copy()

df_out.to_csv(DATA_DIR / 'df_entrenamiento_autor.csv', index=False)

print("=" * 60)
print("VERIFICACIÓN FINAL — df_entrenamiento_autor.csv")
print("=" * 60)
print(f"Filas          : {len(df_out):,}")
print(f"Columnas       : {len(df_out.columns)} ({len(META)} meta + {len(FEATURES_BASE)} base + {len(FEATURES_AUTOR)} autor + 1 target)")
print(f"Fecha mínima   : {df_out['fecha_votacion'].min().date()}")
print(f"NaN en features: {df_out[FEATURES].isna().sum().sum()}")
print(f"Votos distintos: {df_out[TARGET].unique()}")
print(f"Distribución de votos:")
print(df_out[TARGET].value_counts().to_string())
print("=" * 60)

assert len(df_out) == 20608, f"Se esperaban 20.608 filas, hay {len(df_out)}"
assert df_out['fecha_votacion'].min() >= pd.Timestamp('2019-01-01')
assert df_out[FEATURES].isna().sum().sum() == 0, "Hay NaN en las features"
assert not df_out['id_votacion'].isin(IDS_ANOMALOS).any()
print()
print("✓ T10 PASA: dataset de entrenamiento con autoría guardado correctamente.")


VERIFICACIÓN FINAL — df_entrenamiento_autor.csv
Filas          : 20,608
Columnas       : 408 (7 meta + 394 base + 6 autor + 1 target)
Fecha mínima   : 2019-04-04
NaN en features: 0
Votos distintos: ['AFIRMATIVO' 'NEGATIVO' 'ABSTENCIÓN']
Distribución de votos:
voto
AFIRMATIVO    15781
NEGATIVO       4148
ABSTENCIÓN      679

✓ T10 PASA: dataset de entrenamiento con autoría guardado correctamente.
